# Producing initial charts for Mission Studio and Mission Radar

In [1]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import crunchbase
from discovery_utils.utils.io import safe_yaml_load
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts
)

from discovery_utils.utils.llm import batch_check


PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [2]:
CB = crunchbase.CrunchbaseGetter()

2025-02-27 14:45:15,021 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-02-27 14:45:15,204 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-02-24


In [3]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    # "decarbonisation_general",
    "district_heating",
    "energy_efficiency",
    "energy_grid",
    # "energy_storage",
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    # "renewables_general",
    "solar_thermal",
    # "solar",
    # "wind"
]

In [4]:
def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_companies_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return CB.get_companies_in_nesta_categories("topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = CB.get_organisation_text(selected_df)
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
    system_message = batch_check.generate_relevance_check_system_message(config)

    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]

    processor = batch_check.LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.5)    

In [5]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        logging.info(f"Checking relevance for {config_name}")
        config = get_config_dict(config_name)
        selected_df = get_companies_from_config(config)
        await check_relevance(selected_df, config_name, config)
        

In [6]:
await check_all_configs(CONFIG_NAMES)

2025-02-27 14:45:15,228 - root - INFO - Checking relevance for bioenergy
2025-02-27 14:45:15,231 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-02-27 14:45:15,426 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-02-27 14:45:15,427 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-02-27 14:45:15,452 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-02-27 14:45:15,453 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-02-27 14:45:15,454 - botocore.httpchecksum - INFO - Skipping checksum validati